In [13]:
!pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 1.9 MB/s  0:00:40m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.1/684.1 kB 2.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.8 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1━━━━━━━━━━━━━━━━━━ 1/8 [setuptools]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [torchvision] [torchvision]


In [17]:
!pip install -q transformers datasets peft accelerate evaluate scikit-learn

In [7]:
import pandas as pd
import rdkit
import deepchem
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

In [8]:
## IMPORTATION DE LA BASE DE DONNÉES ##

import pandas as pd

# Comme le fichier est dans le même dossier que ton notebook,
# il suffit d'écrire son nom exact.
df = pd.read_csv('df_elec.csv')

print(f"✅ Succès ! {len(df)} molécules chargées.")
display(df.head(10))

✅ Succès ! 534119 molécules chargées.


,Unnamed: 0,smiles,elec_sites,elec_names,MAA_values,elec_GCS_3_cm5,Set
0,0,NOCc1cccc(I)c1,3,double_bond,90.348433,"[-0.01706, 0.12057, -0.11146, -0.08969, 0.0, -...",Train_fold5
1,1,NOCc1cccc(I)c1,4,double_bond,94.924314,"[-0.08969, -0.01706, -0.08484, 0.09939, 0.0, 0...",Train_fold2
2,2,NOCc1cccc(I)c1,5,double_bond,91.330269,"[-0.08484, -0.10499, -0.08969, 0.09122, 0.0, 0...",Train_fold3
3,3,NOCc1cccc(I)c1,6,double_bond,102.683928,"[-0.10499, 0.01492, -0.08484, 0.08707, 0.0, 0....",Train_fold1
4,4,NOCc1cccc(I)c1,7,double_bond,276.204538,"[0.01492, 0.00479, -0.11146, -0.10499, 0.0, 0....",Train_fold3
5,5,NOCc1cccc(I)c1,9,double_bond,106.802353,"[-0.11146, 0.01492, -0.01706, 0.08581, 0.0, 0....",Train_fold2
6,6,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,8,double_bond,59.505283,"[0.72113, -0.68009, -0.74473, -0.73671, 0.0, 0...",Train_fold3
7,7,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,12,double_bond,92.271444,"[-0.03104, 0.09821, 0.05895, -0.07408, 0.0, -0...",Train_fold4
8,8,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,13,double_bond,116.342355,"[-0.07408, -0.03104, -0.10399, 0.10666, 0.0, 0...",Train_fold4
9,9,CN(C)C(=O)CCNC(=O)NCc1ccc(Br)cc1Cl,14,double_bond,96.497045,"[-0.10399, 0.0548, -0.07408, 0.10411, 0.0, -0....",Train_fold2


In [14]:
import torch

if torch.backends.mps.is_available():
    print("🚀 VICTOIRE ! L'accélérateur graphique du M4 (MPS) est prêt à chauffer !")
else:
    print("⚠️ Mince, PyTorch ne trouve pas le GPU. On tourne sur le CPU basique.")

🚀 VICTOIRE ! L'accélérateur graphique du M4 (MPS) est prêt à chauffer !


In [18]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np


print("1️⃣ Préparation des données (Version ChemBERTa)...")

def build_prompt_sans_charges(row):
    # On ne donne que le SMILES et le Site !
    instruction = f"SMILES: {row['smiles']} | Site: {row['elec_sites']} | MAA: "
    return instruction

# On l'applique à ton tableau (assure-toi que df est bien chargé)
df['prompt_question'] = df.apply(build_prompt_sans_charges, axis=1)

# On sépare en Train/Test
train_df = df[df['Set'].str.startswith('Train')]
test_df = df[df['Set'].str.startswith('Test')]

# On convertit pour Hugging Face
train_dataset = Dataset.from_pandas(train_df[['prompt_question', 'MAA_values']])
test_dataset = Dataset.from_pandas(test_df[['prompt_question', 'MAA_values']])

print("✅ Textes prêts ! (Exemple :", df['prompt_question'].iloc[0], ")")

/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1️⃣ Préparation des données (Version ChemBERTa)...
✅ Textes prêts ! (Exemple : SMILES: NOCc1cccc(I)c1 | Site: 3 | MAA:  )


In [19]:
print("2️⃣ Préparation du Tokenizer et du Modèle (ChemBERTa)...")
model_name = "DeepChem/ChemBERTa-77M-MTR" 

tokenizer = AutoTokenizer.from_pretrained(model_name)
# ChemBERTa gère le padding nativement, on n'a plus besoin du hack de GPT-2 !

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=1, 
    problem_type="regression"
)

print("3️⃣ Configuration de LoRA (Adapté pour ChemBERTa)...")
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    # ⚠️ CHANGEMENT CRUCIAL ICI ⚠️
    # ChemBERTa utilise des couches "query" et "value" (et non "c_attn" comme GPT-2)
    target_modules=["query", "value"] 
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

2️⃣ Préparation du Tokenizer et du Modèle (ChemBERTa)...


[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `199`.
Loading weights: 100%|██████████| 53/53 [00:00<00:00, 25107.08it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                        | Status     | 
---------------------------+------------+-
norm_std                   | UNEXPECTED | 
norm_mean                  | UNEXPECTED | 
regression.dense.weight    | UNEXPECTED | 
regression.out_proj.weight | UNEXPECTED | 
regression.out_proj.bias   | UNEXPECTED | 
regression.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trainin

3️⃣ Configuration de LoRA (Adapté pour ChemBERTa)...
trainable params: 185,089 || all params: 3,612,914 || trainable%: 5.1230


In [20]:
print("4️⃣ Tokenization des textes...")
def tokenize_function(examples):
    # On transforme le texte en nombres pour le modèle
    return tokenizer(examples["prompt_question"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# On renomme 'MAA_values' en 'labels' car le Trainer exige ce nom
tokenized_train = tokenized_train.rename_column("MAA_values", "labels")
tokenized_test = tokenized_test.rename_column("MAA_values", "labels")


print("5️⃣ Définition des métriques de succès...")
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Calculs classiques
    r2 = r2_score(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    
    # Calcul du RMSE relatif (en pourcentage)
    moyenne_maa = np.mean(labels)
    rrmse_pourcentage = (rmse / moyenne_maa) * 100
    
    return {
        "r2": r2, 
        "rmse": rmse, 
        "rrmse_%": rrmse_pourcentage
    }

print("6️⃣ Lancement de l'Entraînement ! 🚀")
training_args = TrainingArguments(
    output_dir="./resultats_chemberta",
    learning_rate=2e-4,
    per_device_train_batch_size=8, # 👈 Baisse à 8 (au lieu de 16) pour soulager le Mac
    per_device_eval_batch_size=8,  # 👈 Baisse à 8 ici aussi
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=False, # 👈 Mets False. Le FP16 plante parfois sur les Mac M1/M2.
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

print("✅ Entraînement terminé ! Voici l'évaluation finale :")
print(trainer.evaluate())

4️⃣ Tokenization des textes...


Map: 100%|██████████| 80118/80118 [00:02<00:00, 38736.62 examples/s]
/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


5️⃣ Définition des métriques de succès...
6️⃣ Lancement de l'Entraînement ! 🚀


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,R2,Rmse,Rrmse %
1,4748.195312,3829.587402,0.257744,61.883656,62.542270
2,3177.112305,3750.506348,0.273071,61.241378,61.893156
3,4774.304297,3726.271240,0.277769,61.043192,61.692861


/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


✅ Entraînement terminé ! Voici l'évaluation finale :


/opt/anaconda3/envs/chem_clean/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,R2,Rmse,Rrmse %
4774.304297,3726.271240,3,0.277769,61.043192,61.692861


{'eval_loss': 3726.271240234375, 'eval_r2': 0.2777685523033142, 'eval_rmse': 61.04319159606889, 'eval_rrmse_%': 61.69286064584494}
